In [8]:
# ============================================================
# 2. Imports
# ============================================================

import re
import numpy as np
import pandas as pd
import requests
import chromadb

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# ============================================================
# 3. Load Document
#    Wikipedia: Artificial Intelligence
# ============================================================

with open("ai_document.txt", "r", encoding="utf-8") as file:
    document_text = file.read()

print("Document loaded successfully")
print("Words:", len(document_text.split()))
print("Characters:", len(document_text))

Document loaded successfully
Words: 361
Characters: 2680


In [10]:
# ============================================================
# 4. Text Chunking
#    Required: chunk_size=300, overlap=50
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_text(document_text)

chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]

print("Total chunks:", len(chunks))

Total chunks: 13


In [11]:
# ============================================================
# 5. Generate Embeddings
# ============================================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.23it/s]

Embedding shape: (13, 384)


In [12]:
# ============================================================
# 6. Store Embeddings in ChromaDB
# ============================================================

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="ai_document"
)

collection.add(
    ids=chunk_ids,
    documents=chunks,
    embeddings=embeddings.tolist()
)

print("ChromaDB documents:", collection.count())

ChromaDB documents: 13


In [13]:
# ============================================================
# 7. BM25 Keyword Search
# ============================================================

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


tokenized_chunks = [
    tokenize(chunk)
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)


def bm25_search(query, top_k=10):

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(indices, 1):

        results.append({
            "id": chunk_ids[index],
            "text": chunks[index],
            "rank": rank,
            "score": float(scores[index])
        })

    return results


# ============================================================
# 8. Vector Search
# ============================================================

def vector_search(query, top_k=10):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    output = []

    for rank, (doc_id, text) in enumerate(
        zip(
            results["ids"][0],
            results["documents"][0]
        ),
        1
    ):

        output.append({
            "id": doc_id,
            "text": text,
            "rank": rank
        })

    return output


# ============================================================
# 9. Reciprocal Rank Fusion (RRF)
# ============================================================

def reciprocal_rank_fusion(
    vector_results,
    bm25_results,
    k=60
):

    scores = {}
    documents = {}

    for result in vector_results:

        doc_id = result["id"]

        scores[doc_id] = scores.get(doc_id, 0) + (
            1 / (k + result["rank"])
        )

        documents[doc_id] = result["text"]


    for result in bm25_results:

        doc_id = result["id"]

        scores[doc_id] = scores.get(doc_id, 0) + (
            1 / (k + result["rank"])
        )

        documents[doc_id] = result["text"]


    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        {
            "id": doc_id,
            "text": documents[doc_id],
            "rrf_score": score
        }
        for doc_id, score in ranked
    ]

# ============================================================
# 10. Hybrid Search
# ============================================================

def hybrid_search(query, top_k=10):

    vector_results = vector_search(query, top_k)
    bm25_results = bm25_search(query, top_k)

    fused_results = reciprocal_rank_fusion(
        vector_results,
        bm25_results
    )

    return fused_results[:top_k]


# ============================================================
# 11. Cross-Encoder Reranking
# ============================================================

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


def rerank_results(query, results, top_k=3):

    pairs = [
        [query, result["text"]]
        for result in results
    ]

    scores = reranker.predict(pairs)

    for result, score in zip(results, scores):
        result["cross_encoder_score"] = float(score)

    results.sort(
        key=lambda x: x["cross_encoder_score"],
        reverse=True
    )

    return results[:top_k]


# ============================================================
# 12. Ollama LLM
# ============================================================

llm = ChatOllama(
    model="qwen3:0.6b",
    base_url="http://localhost:11434"
)


# ============================================================
# 13. LangChain RAG Chain
# ============================================================

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Answer using ONLY the context.

If the answer is not found in the context,
say exactly: "Not in context."

Do not use outside knowledge.

Context:
{context}
"""
    ),
    (
        "human",
        "{question}"
    )
])

rag_chain = prompt | llm | StrOutputParser()


# ============================================================
# 14. Complete RAG Pipeline
# ============================================================

def answer_question(query):

    # Vector + BM25 + RRF
    hybrid_results = hybrid_search(
        query,
        top_k=10
    )

    # Cross-encoder → Top 3
    final_results = rerank_results(
        query,
        hybrid_results,
        top_k=3
    )

    # Context
    context = "\n\n".join(
        f"[Chunk {i+1}]\n{result['text']}"
        for i, result in enumerate(final_results)
    )

    # LLM
    answer = rag_chain.invoke({
        "context": context,
        "question": query
    })

    return answer, final_results


# ============================================================
# 15. Test with 5 Questions
# ============================================================

questions = [
    # General question
    "What is artificial intelligence?",

    # Semantic understanding
    "How can computers perform tasks that normally require human intelligence?",

    # BM25 / exact keyword matching
    "What does the article say about artificial general intelligence?",

    # Another semantic question
    "How does machine learning relate to artificial intelligence?",

    # History
    "What does the article say about the history of artificial intelligence?"
]

for i, question in enumerate(questions, 1):

    print("\n" + "=" * 70)
    print(f"QUESTION {i}")
    print("=" * 70)

    print(question)

    answer, retrieved_chunks = answer_question(question)

    print("\nANSWER:")
    print(answer)

    print("\nTOP 3 RETRIEVED CHUNKS:")

    for j, chunk in enumerate(retrieved_chunks, 1):

        print(
            f"\nChunk {j} "
            f"(RRF={chunk['rrf_score']:.4f}, "
            f"CrossEncoder={chunk['cross_encoder_score']:.4f})"
        )

        print(chunk["text"])



Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 5618.95it/s]



QUESTION 1
What is artificial intelligence?

ANSWER:
Not in context.

TOP 3 RETRIEVED CHUNKS:

Chunk 1 (RRF=0.0328, CrossEncoder=10.7966)
Artificial intelligence (AI) is intelligence demonstrated by machines,
as opposed to the natural intelligence displayed by humans and other animals.

Chunk 2 (RRF=0.0323, CrossEncoder=8.8036)
Artificial intelligence can be divided into different categories.
Artificial narrow intelligence refers to systems designed to perform
specific tasks. Artificial general intelligence refers to a hypothetical
machine capable of understanding and performing a wide range of intellectual

Chunk 3 (RRF=0.0306, CrossEncoder=7.3810)
Artificial intelligence was founded as an academic discipline in 1956.
The field went through multiple cycles of optimism, followed by periods
of disappointment and loss of funding, known as AI winters.

QUESTION 2
How can computers perform tasks that normally require human intelligence?

ANSWER:
The answer is found in the context. compute

In [16]:
# ============================================================
# 16. Retrieval Comparison
# ============================================================

query = "How do computers learn from data?"

vector_results = vector_search(query, 5)
bm25_results = bm25_search(query, 5)
hybrid_results = hybrid_search(query, 10)

print("\nVECTOR SEARCH")
for result in vector_results:
    print(result["rank"], result["id"])

print("\nBM25 SEARCH")
for result in bm25_results:
    print(result["rank"], result["id"])

print("\nHYBRID RRF SEARCH")
for rank, result in enumerate(hybrid_results, 1):
    print(rank, result["id"], result["rrf_score"])


VECTOR SEARCH
1 chunk_4
2 chunk_7
3 chunk_6
4 chunk_10
5 chunk_5

BM25 SEARCH
1 chunk_4
2 chunk_7
3 chunk_12
4 chunk_6
5 chunk_11

HYBRID RRF SEARCH
1 chunk_4 0.03278688524590164
2 chunk_7 0.03225806451612903
3 chunk_6 0.03149801587301587
4 chunk_12 0.030798389007344232
5 chunk_10 0.030776515151515152
6 chunk_5 0.029877369007803793
7 chunk_11 0.029877369007803793
8 chunk_0 0.015151515151515152
9 chunk_9 0.014925373134328358
10 chunk_2 0.014705882352941176
